In [ ]:
# 1-dataset model (HTCas9)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = -0.21712553841179372
../../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = -0.2530038657957296
../../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = -0.207739727913278
../../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = -0.19332582209500834
../../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = -0.19596996156446936
../../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = -0.21149806562413345
../../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = -0.19348689361776222
../../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = -0.2380472004145123
../../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = -0.2228942713168685
../../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spea

In [ ]:
# 2-dataset model (HTCas9+HT11)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = -0.25677004554608646
../../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = -0.23896588039530667
../../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = -0.24126071979056812
../../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = -0.24633438411421993
../../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = -0.23239832198640317
../../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = -0.2303291367991023
../../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = -0.2351158852210229
../../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = -0.2531820959631021
../../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = -0.291808082350878
../../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spea

In [ ]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.41363768097855375
../../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.4330038342251124
../../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.6268459217609387
../../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.5845863446166275
../../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.47532178690302823
../../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.22510688646350266
../../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.18621194508755762
../../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.5196991102213839
../../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.5975137027877262
../../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.

In [ ]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.6229546473679752
../../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.6540992289996429
../../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.5729814597599283
../../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.3845387196121785
../../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.5702366689855467
../../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.5176418090744148
../../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.27390400722329405
../../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.3790439880420639
../../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = -0.13758672277814293
../../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.2

In [ ]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.38736242065278204
../../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.33207268664835127
../../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.34050133002541894
../../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.5687190825408953
../../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.5540274856751864
../../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.48088029943369837
../../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.5385714735469467
../../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.5977061059880796
../../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.5736479851588754
../../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.

In [ ]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc4():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc4():
    with open('filtered_ratio_NM_002794_4.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc4 = load_branch1_data_RfxCas13d_mmc4()
    X1_RfxCas13d_mmc4   = np.asarray(X1_RfxCas13d_mmc4)
    X1 = np.concatenate([X1_RfxCas13d_mmc4], axis=0) 

    rates_RfxCas13d_mmc4 = load_reaction_rates_RfxCas13d_mmc4()
    rates_RfxCas13d_mmc4   = np.asarray(rates_RfxCas13d_mmc4)
    rates = np.concatenate([rates_RfxCas13d_mmc4], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc4 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc4 = np.arange(len(rates_RfxCas13d_mmc4))
    selected_indices_RfxCas13d_mmc4 = np.random.choice(len(full_indices_RfxCas13d_mmc4), size=len(full_indices_RfxCas13d_mmc4), replace=False)
    unseen_indices_RfxCas13d_mmc4 = np.setdiff1d(full_indices_RfxCas13d_mmc4, selected_indices_RfxCas13d_mmc4)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc4)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc4 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc4)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc4, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.25897024489404685
../../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.3622466533626646
../../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.6059072278603803
../../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.42149992085062876
../../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.5445239034152101
../../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.627091638393285
../../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.6401781883599081
../../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.6448642486094907
../../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.541261420239386
../../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.4992